# Area affected by droughts (time series)


In [ ]:
# general
import pandas as pd
import numpy as np

# plotting
import plotly.graph_objects as go
from   plotly.subplots import make_subplots
from PIL import Image
import io

from src.utils import load_config
from src.plotting import base_layout, FONT_FAMILY, PANEL_LABEL_COLOR, DEFAULT_MARGIN, scenario_colors, save_figure
cfg = load_config()

scenarios = [s["name"] for s in cfg["scenarios"]]
colors    = scenario_colors(cfg)

## Data

In [ ]:
# data from drought analysis
df = pd.read_csv("data/hydro/droughts_hydroatlas_area_ts.csv", index_col=["scenario", "date"], parse_dates=["date"])
df = df.rolling(window=90, min_periods=1, center=True).mean()
df = df.rename(
    index={"historical": "Historical",
            "15C": scenarios[0], 
            "28C": scenarios[1], 
            "40C": scenarios[2]},
    level=0,
)

# regional values 
df_agg = pd.DataFrame(index=df.index)
for region, (lat_max, lat_min) in cfg["regions"].items():
    cols = [str(c) for c in df.columns if lat_min <= int(c) < lat_max]
    df_agg[region] = df[cols].sum(axis=1)

# total values (for animation)
df_all = df_agg.sum(axis=1)
df_all = df_all.reset_index().pivot_table(index='date', columns='scenario')
df_all.columns = df_all.columns.get_level_values(1)

for scenario in scenarios:
    df_all[scenario] = ((df_all[scenario] / df_all["Historical"]) -1)*100

## Plot (MS)

In [ ]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True)

# Manually define domains: [bottom, top] in paper coords (0 to 1)
fig.layout.yaxis.domain  = [0.85, 1.00]  # row 1 (small)
fig.layout.yaxis2.domain = [0.52, 0.84]  # row 2 (large)

fig.layout.yaxis3.domain = [0.33, 0.48]  # row 3 (small) — bigger gap above
fig.layout.yaxis4.domain = [0.00, 0.32]  # row 4 (large)

# Build categorical x-axis (daily as strings) and yearly tick labels
date_idx = df_agg[region].loc["Historical"].index
cat_array = date_idx.strftime('%Y-%m-%d').tolist()
year_idx = pd.date_range(date_idx.min().replace(month=1, day=1),
                         date_idx.max().replace(month=1, day=1),
                         freq='YS')
year_tickvals = year_idx.strftime('%Y-%m-%d').tolist()
year_labels = year_idx.strftime('%Y').tolist()

# Row 1: Whole Andes - Historical extent
fig.add_trace(
    go.Scatter(
        x=pd.to_datetime(df_all.index).strftime('%Y-%m-%d'),
        y=df_all["Historical"] / 1000,
        mode='lines',
        fill='tozeroy',
        fillcolor='rgba(29, 28, 28, 0.3)', 
        line=dict(width=1, color="#1D1C1C"),
        showlegend=False,
    ),
    row=1, col=1
)

# Row 2: Whole Andes - What-if stacked areas
base = np.zeros_like(df_all[scenarios[0]].values)
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    if i == 0:
        upper = df_all[scenario].values
    else:
        upper = df_all[scenario].values - df_all[scenarios[i-1]].values

    fig.add_trace(
        go.Scatter(
            x=df_all.index.strftime('%Y-%m-%d'),
            y=base + upper,
            name=scenario,
            mode='lines',
            fill='tonexty',
            line=dict(color=color, width=0.5),
            fillcolor=f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, 1)",
            showlegend=True,
        ),
        row=2, col=1
    )
    base += upper

# Row 3: Central Andes - Historical extent
fig.add_trace(
    go.Scatter(
        x=df_agg.loc["Historical"].index.strftime('%Y-%m-%d'),
        y=df_agg.loc["Historical"]["Central Andes"] / 1000,
        mode='lines',
        fill='tozeroy',
        fillcolor='rgba(29, 28, 28, 0.3)', 
        line=dict(width=1, color="#1D1C1C"),
        showlegend=False,
    ),
    row=3, col=1
)

# Row 4: Central Andes - What-if scenario lines
base = np.zeros_like(df_agg['Central Andes'].loc[scenarios[0]].values)
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    diff = (df_agg['Central Andes'].loc[scenario].values - df_agg['Central Andes'].loc['Historical'].values) / 100
    if i == 0:
        upper = diff
    else:
        prev_diff = (df_agg['Central Andes'].loc[scenarios[i-1]].values - df_agg['Central Andes'].loc['Historical'].values) / 100
        upper = diff - prev_diff

    fig.add_trace(
        go.Scatter(
            x=df_agg['Central Andes'].loc[scenario].index.strftime('%Y-%m-%d'),
            y=base + upper,
            name=scenario,
            mode='lines',
            fill='tonexty',
            line=dict(color=color, width=0.5),
            fillcolor=f"rgba({int(color[1:3], 16)}, {int(color[3:5], 16)}, {int(color[5:7], 16)}, 1)",
            showlegend=False,
        ),
        row=4, col=1
    )
    base += upper

# y-axes
fig.update_yaxes(range=[0, 800], dtick = 400, title_text="Extent (10³ km²)", title_standoff=0, row=1, col=1)
fig.update_yaxes(range=[0, 60], title_text="What-if change (%)", title_standoff=0, side="right", row=2, col=1)
fig.update_yaxes(range=[0, 100], title_text="Extent (10³ km²)", title_standoff=0, row=3, col=1)
fig.update_yaxes(range=[0, 150], title_text="What-if change (%)", title_standoff=0, side="right", row=4, col=1)

# x-axes
fig.update_xaxes(
    type='category',
    categoryorder='array',
    categoryarray=cat_array,
    tickmode='array',
    tickvals=year_tickvals[::2],
    ticktext=year_labels[::2],
)

# Annotations
annotations = [
    ("a) Whole Andes (8–56ºS)", 0.99, 0.995),
    ("b) Central Andes (30–35ºS)", 0.99, 0.48),
]

for text, x, y in annotations:
    label, _, rest = text.partition(") ")
    fig.add_annotation(text=f"<b>{label})</b> {rest}", x=x, y=y, xref="paper", yref="paper",
                       showarrow=False, font=dict(size=13, color=PANEL_LABEL_COLOR),
                       xanchor="right", yanchor="top")

# Megadrought vertical line in rows 3 and 4
for yref in ["y3 domain", "y4 domain"]:
    fig.add_shape(type="line",
                  x0="2010-01-01", x1="2010-01-01",
                  y0=0, y1=1,
                  xref="x", yref=yref,
                  line=dict(color="grey", width=1, dash="dot"))

# Arrow pointing to the future (row 3, mid-height)
fig.add_annotation(
    x="2010-02-01", y=0.91,
    ax="2010-02-01", ay=0.91,
    xref="x", yref="y4 domain",
    axref="x", ayref="y4 domain",
    text="⇒ Megadrought", showarrow=True,
    font=dict(size=14),
    xanchor="left",
)

fig.update_layout(**base_layout(
    legend=dict(title="Scenarios:", orientation="h", y=1.05, x=0.4, bgcolor="rgba(255,255,255,0)"),
    autosize=False, width=800, height=600, barmode='overlay',
    margin = dict(l=50, r=50, b=50, t=10),
    font=dict(size=12),
))

save_figure(fig, "figures/fig_whatif_drought_extent.svg", height=600, width=900)
fig.show()

## Plot (animation)

In [ ]:
# Build categorical x-axis (daily as strings) and yearly tick labels
date_idx = df_all.index
cat_array = date_idx.strftime('%Y-%m-%d').tolist()
year_idx = pd.date_range(date_idx.min().replace(month=1, day=1),
                         date_idx.max().replace(month=1, day=1),
                         freq='YS')
year_tickvals = year_idx.strftime('%Y-%m-%d').tolist()
year_labels = year_idx.strftime('%Y').tolist()

# Store images for GIF
images = []

# Panel 2: Stacked area plot for scenarios
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.03, row_heights=[0.4, 0.6],
                    shared_xaxes=True, shared_yaxes=False)

# Panel 1: Historical drought extent
fig.add_trace(go.Scatter(
    x=df_all.index.strftime('%Y-%m-%d'),
    y=df_all["Historical"] / 1000,
    mode='lines', 
    line=dict(width=1, color="#2E2D2D"),
    fill='tozeroy',
    fillcolor='rgba(40, 40, 40, 0.2)',
    showlegend=False,
), row=1, col=1)

# Panel 2: Stacked scenario changes
base = np.zeros_like(df_all[scenarios[0]].values)
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    # Calculate the incremental change for this scenario
    if i == 0:
        upper = df_all[scenario].values
    else:
        upper = df_all[scenario].values - df_all[scenarios[i-1]].values

    fig.add_trace(go.Scatter(
        x=pd.to_datetime(df_all.index.values).strftime('%Y-%m-%d'),
        y=base + upper,
        name=scenario,
        mode='lines',
        fill='tonexty',
        line=dict(color=color, width=1),
        fillcolor=color,
        showlegend=True,
    ), row=2, col=1)
    base = base + upper

    # x-axes
    fig.update_xaxes(
        type='category',
        categoryorder='array',
        categoryarray=cat_array,
        tickmode='array',
        ticks = "outside",
        tickvals=year_tickvals[::2],
        ticktext=year_labels[::2],
        griddash="dot")

    fig.update_yaxes(title = "Drought extent (10³ km²)", range = [0, 300], row = 1, col = 1)
    fig.update_yaxes(title = "What-if change (%)", range = [0, 100],  row = 2, col = 1)
    fig.update_yaxes(ticks="outside", griddash="dot")

    # Layout with increased bottom margin for x-axis labels
    fig.update_layout(font_size = 14, font_family = FONT_FAMILY, 
        height=600, width=1100,
        template="plotly",
        margin=DEFAULT_MARGIN,  # Increase bottom margin
        legend=dict(bgcolor='rgba(255,255,255,0)',
            title= dict(text="Scenarios:"),
            yanchor="bottom", y=0.40,
            xanchor="right", x=0.93,
        )
    )
    
    # Convert figure to image and add to list
    img_bytes = fig.to_image(format="png", scale=4, height=600, width=1100)
    img = Image.open(io.BytesIO(img_bytes))
    images.append(img)

# Create GIF
images[0].save(
    'figures/fig_whatif_drought_extent_all.gif',
    save_all=True,
    append_images=images[1:],
    duration=2000,  # milliseconds per frame
    loop=0
)

In [ ]:
# Build categorical x-axis (daily as strings) and yearly tick labels
date_idx = df_agg.loc["Historical"].index
cat_array = date_idx.strftime('%Y-%m-%d').tolist()
year_idx = pd.date_range(date_idx.min().replace(month=1, day=1),
                         date_idx.max().replace(month=1, day=1),
                         freq='YS')
year_tickvals = year_idx.strftime('%Y-%m-%d').tolist()
year_labels = year_idx.strftime('%Y').tolist()

# Store images for GIF
images = []

# Panel 2: Stacked area plot for scenarios
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.03, row_heights=[0.4, 0.6],
                    shared_xaxes=True, shared_yaxes=False)

# Panel 1: Central Andes - Historical drought extent
fig.add_trace(go.Scatter(
    x=df_agg.loc["Historical"].index.strftime('%Y-%m-%d'),
    y=df_agg.loc["Historical"]["Central Andes"] / 1000,
    mode='lines', 
    line=dict(width=1, color="#2E2D2D"),
    fill='tozeroy',
    fillcolor='rgba(40, 40, 40, 0.2)',
    showlegend=False,
), row=1, col=1)

# Panel 2: Central Andes - Stacked scenario changes
base = np.zeros_like(df_agg['Central Andes'].loc[scenarios[0]].values)
for i, (scenario, color) in enumerate(zip(scenarios, colors)):
    diff = (df_agg['Central Andes'].loc[scenario].values - df_agg['Central Andes'].loc['Historical'].values) / 100
    if i == 0:
        upper = diff
    else:
        prev_diff = (df_agg['Central Andes'].loc[scenarios[i-1]].values - df_agg['Central Andes'].loc['Historical'].values) / 100
        upper = diff - prev_diff

    fig.add_trace(go.Scatter(
        x=df_agg['Central Andes'].loc[scenario].index.strftime('%Y-%m-%d'),
        y=base + upper,
        name=scenario,
        mode='lines',
        fill='tonexty',
        line=dict(color=color, width=1),
        fillcolor=color,
        showlegend=True,
    ), row=2, col=1)
    base += upper

    # x-axes
    fig.update_xaxes(
        type='category',
        categoryorder='array',
        categoryarray=cat_array,
        tickmode='array',
        ticks = "outside",
        tickvals=year_tickvals[::2],
        ticktext=year_labels[::2],
        griddash="dot")

    fig.update_yaxes(title = "Drought extent (10³ km²)", range = [0, 70], row = 1, col = 1)
    fig.update_yaxes(title = "What-if change (%)", range = [0, 70], row = 2, col = 1)
    fig.update_yaxes(ticks="outside", griddash="dot")

    # Layout with increased bottom margin for x-axis labels
    fig.update_layout(font_size = 14, font_family = FONT_FAMILY, 
        height=600, width=1100,
        template="plotly",
        margin=DEFAULT_MARGIN,
        legend=dict(bgcolor='rgba(255,255,255,0)',
            title= dict(text="Scenarios:"),
            yanchor="bottom", y=0.40,
            xanchor="right", x=0.13,
        )
    )
    
    # Convert figure to image and add to list
    img_bytes = fig.to_image(format="png", scale=4, height=600, width=1100)
    img = Image.open(io.BytesIO(img_bytes))
    images.append(img)

# Create GIF
images[0].save(
    'figures/fig_whatif_drought_extent_ca.gif',
    save_all=True,
    append_images=images[1:],
    duration=2000,  # milliseconds per frame
    loop=0
)
